In [7]:
# Package 
import pandas as pd
from pathlib import Path
from feast import FeatureStore

In [8]:
# ----------------------------
# 0) Config
# ----------------------------
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO","M2SL","OILPRICEX",
    "RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

# Feast repo (robuste : indépendant du dossier courant)
REPO_PATH = (
    next(
        p for p in [Path.cwd()] + list(Path.cwd().parents)
        if (p / "2_data_processing").exists()
    )
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

store = FeatureStore(repo_path=str(REPO_PATH))
print("Project:", store.project)
print("Feature views:", [fv.name for fv in store.list_feature_views()])

Project: unemployment_feature_store
Feature views: ['stationary_value', 'raw_value']


In [9]:
# ----------------------------
# 1) Construire les dates (SANS DATES_PATH)
#    -> on génère un calendrier puis Feast filtrera ce qui existe réellement
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

# Pour récupérer une liste de dates "validées" par Feast :
# on interroge UNRATE seulement, puis on prend les dates retournées.
entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})

df_unrate = store.get_historical_features(
    entity_df=entity_df_unrate,
    features=["raw_value:value"],          # juste pour valider les dates existantes
    full_feature_names=True,
).to_df()

# Normalisation date + tri + unique
dates = (
    pd.to_datetime(df_unrate["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)                 # enlève le +00:00
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00


In [10]:
# ----------------------------
# 2) Entity DF multi-séries
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

print("entity_df shape:", entity_df.shape)
print(entity_df.head())

entity_df shape: (8811, 2)
  series_id       date
0  BUSLOANS 1959-01-01
1  BUSLOANS 1959-02-01
2  BUSLOANS 1959-03-01
3  BUSLOANS 1959-04-01
4  BUSLOANS 1959-05-01


In [11]:
# ----------------------------
# 3) Fetch stationary features (Feast)
# ----------------------------
df_stationary = store.get_historical_features(
    entity_df=entity_df,
    features=["stationary_value:value"],
    full_feature_names=True,               # IMPORTANT
).to_df()

# Normaliser types
df_stationary["date"] = pd.to_datetime(df_stationary["date"], utc=True, errors="coerce").dt.tz_convert(None)

# Colonne valeur robuste (au cas où)
value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    # fallback (rare)
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Colonne attendue '{value_col}' absente. Trouvé: {candidates}")

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date  stationary_value__value
0  BUSLOANS 1960-01-01                 0.011578
1    INDPRO 1960-01-01                 0.091976
2     USREC 1960-01-01                 0.000000
3      M2SL 1960-01-01                 0.001323
4  CPIAUCSL 1960-01-01                -0.006156


In [12]:
# ----------------------------
# 4) Pivot LONG → WIDE (1 colonne par série)
# ----------------------------
df_wide = (
    df_stationary
    .pivot_table(index="date", columns="series_id", values=value_col, aggfunc="last")
    .sort_index()
)

# (Optionnel) réordonner les colonnes selon series_ids
df_wide = df_wide.reindex(columns=series_ids)

print("df_wide shape:", df_wide.shape)
df_wide.head()

df_wide shape: (789, 11)


series_id,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,UNRATE,USREC
date,,,,,,,,,,,
1960-01-01,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,-0.8,0.0
1960-02-01,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,-1.1,0.0
1960-03-01,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,-0.2,0.0
1960-04-01,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0,0.0
1960-05-01,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,0.0,1.0


# Vérifier le Nan

In [13]:
df_wide.isna().sum()

series_id
BUSLOANS           0
CPIAUCSL           0
DPCERA3M086SBEA    1
INDPRO             1
M2SL               0
OILPRICEX          0
RPI                1
SP500              0
TB3MS              0
UNRATE             0
USREC              0
dtype: int64

In [16]:
# df = ton DataFrame wide, index datetime (MS), colonnes = series_id

cols_with_nan = ["DPCERA3M086SBEA", "INDPRO", "RPI"]

for col in cols_with_nan:
    nan_dates = df_wide.index[df_wide[col].isna()]
    print(f"\n🔎 {col}")
    print(f"Nombre de NaN : {len(nan_dates)}")
    print("Dates :", list(nan_dates))


🔎 DPCERA3M086SBEA
Nombre de NaN : 1
Dates : [Timestamp('2025-09-01 00:00:00')]

🔎 INDPRO
Nombre de NaN : 1
Dates : [Timestamp('2025-09-01 00:00:00')]

🔎 RPI
Nombre de NaN : 1
Dates : [Timestamp('2025-09-01 00:00:00')]


In [20]:
df_wide

series_id,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,UNRATE,USREC
date,,,,,,,,,,,
1960-01-01,0.011578,-0.006156,0.001204,0.091976,0.001323,0.000000,0.020977,0.017909,0.30,-0.8,0.0
1960-02-01,0.011905,-0.003767,0.006009,0.076960,0.002007,0.000000,0.014565,-0.025663,-0.19,-1.1,0.0
1960-03-01,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.000000,0.006250,-0.070857,-1.18,-0.2,0.0
1960-04-01,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.000000,0.006489,-0.040442,-1.12,0.0,0.0
1960-05-01,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.000000,0.007747,-0.010090,-0.67,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-05-01,0.044487,-0.007941,0.007758,0.000360,0.004820,-0.162581,0.008190,-0.038448,0.03,0.2,0.0
2025-06-01,0.035593,-0.000435,0.002578,0.005179,0.004174,0.026151,0.001403,0.059087,0.03,0.0,0.0
2025-07-01,-0.001303,0.001775,0.005014,0.001050,-0.001062,0.249194,-0.003535,0.159259,0.04,0.0,0.0
